In [2]:
import numpy as np
import random
import math

In [16]:
# Example objective function:
def objective_function(x):
    """Example objective function (Sphere function)."""
    return np.sum(x**2)

#Rastrigin-2 min=[0,0] value=0
def rastrigin2d(X):
    x1, x2 = X
    Z = (x1**2 - 10 * np.cos(2 * np.pi * x1)) + (x2**2 - 10 * np.cos(2 * np.pi * x2)) + 20
    return Z

In [4]:
class Firefly:
    """Represents a single firefly in the Firefly Algorithm."""
    def __init__(self, objective_function, bounds, alpha=0.2, beta_0=0.2, gamma=1.0):
        """
        Initializes a firefly.
        Args|Parameters
            dim (int): Dimension of objective function, calculated from bounds
            bounds (list of tuples): NP Array of (lower_bound, upper_bound) tuples for each dimension.
            alpha (float): Random number parameter.
            beta_0 (float): Minimum attractiveness.
            gamma (float): Absorption coefficient.
            position: Current location
            intensity: Intensity of the light
        """
        self.objective_function = objective_function
        self.bounds = bounds
        self.alpha = alpha
        self.beta_0 = beta_0
        self.gamma = gamma
        self.position = np.array([random.uniform(low, high) for low, high in bounds])
        self.intensity = self.calculate_intensity()

    def display(self):
        """Display the Firefly position"""
        return f"Position: {self.position}, Intensity: {self.intensity}"
        
        
    def calculate_intensity(self):
        """Calculates the brightness of the firefly based on the objective function."""
        return self.objective_function(self.position)
    
    def update_position(self,nearby_firefly):
        """Update position based on distance, clip any over bounds"""
        brightest_firefly = max(nearby_firefly, key=lambda f: f.intensity)
        r = np.linalg.norm(self.position - brightest_firefly.position)
        beta = self.beta_0 * np.exp(-self.gamma * r**2)
        self.position += beta * (brightest_firefly.position - self.position) + self.alpha * (np.random.rand(len(self.position)) - 0.5)
        self.position = np.clip(self.position, [low for low, high in self.bounds], [high for low, high in self.bounds])
        self.intensity = self.calculate_intensity()


In [11]:
#Test the Firefly Class
bounds = np.array([(-5, 5), (-5, 5)])
dim=bounds.shape[0] #dim derived from bounds
firefly=Firefly(objective_function,bounds)
firefly.display()


'Position: [-2.94641282 -3.406191  ], Intensity: 20.283485651368245'

In [28]:
class Firefly_Optimization:
    
    def __init__(self, objective_fn, bounds, num_fireflies=10, 
                 iterations=10, alpha=1., beta_0=1., 
                 gamma=.01, is_minimization=True):
        """
        Initializes the Firefly Optimization algorithm

        num_fireflies: The number of fireflies used
        bounds: Limit of each dimension, numpy array
        dim: number of dimensions calculated from bounds
        iterations: Maximum number of iterations
        alpha: Randomness parameter
        beta_0: Minimum attractiveness
        gamma: absorption coefficient
        
        objection_function: function to optimize
        best_position: None
        best_intensity: Infinity
        
        fireflies: The population of fireflies
        """
        self.objective_fn = objective_fn
        self.bounds = bounds
        self.num_fireflies = num_fireflies
        self.iterations = iterations
        self.alpha = alpha
        self.beta_0 = beta_0
        self.gamma = gamma
        self.is_minimization = is_minimization

        self.best_position = None
        self.best_intensity = float('inf')
        self.fireflies = []


    def generate_population(self):
        for _ in range(self.num_fireflies):
            firefly = Firefly(self.objective_fn, self.bounds, self.alpha, self.beta_0, self.gamma)
            self.fireflies.append(firefly)
            if firefly.intensity < self.best_intensity:
                self.best_intensity = firefly.intensity
                self.best_position = firefly.position


    def print_population(self):
        print(f"Current Population of Fireflies:")
        for firefly in self.fireflies:
            print(firefly.display())

    def optimize(self):
        """Update the position of each firefly to the brighter firefly depending on problem
                            
                    > if minimization problem
                    < if maximization problem
        """
        if self.is_minimization:
            self.fireflies.sort(key=lambda f: f.intensity)
            for i in range(self.iterations):
                for firefly in self.fireflies:
                    firefly.update_position(self.fireflies)
                    if firefly.intensity < self.best_intensity:
                        self.best_intensity = firefly.intensity
                        self.best_position = firefly.position
        else:
            self.fireflies.sort(key=lambda f: f.intensity, reverse=True)
            for i in range(self.iterations):
                for firefly in self.fireflies:
                    firefly.update_position(self.fireflies)
                    if firefly.intensity > self.best_intensity:
                        self.best_intensity = firefly.intensity
                        self.best_position = firefly.position
        return self.best_position, self.best_intensity


    def get_optima(self):
        return self.best_position,self.best_intensity

In [29]:
# Example usage:
FA=Firefly_Optimization(rastrigin2d,bounds,num_fireflies=100, iterations=50)
FA.generate_population()
FA.print_population()

Current Population of Fireflies:
Position: [ 1.4462336  -2.66368237], Intensity: 43.78307448836189
Position: [1.12839198 1.51887762], Intensity: 26.591225009935474
Position: [ 4.18925034 -2.29819977], Intensity: 42.08894048986223
Position: [-2.02578299  0.25181773], Intensity: 14.412349759774138
Position: [-0.13550551 -4.88373787], Intensity: 29.83156509181403
Position: [-0.88030349 -4.81594251], Intensity: 32.63975192760123
Position: [-4.06840506 -3.69954533], Intensity: 44.26539822977179
Position: [2.98207347 3.48871687], Intensity: 41.10215524591954
Position: [-2.71248539 -3.91073375], Intensity: 36.518869247298156
Position: [4.98949318 3.79703232], Intensity: 46.421974814316805
Position: [-2.56519452 -2.01867585], Intensity: 29.8967293486097
Position: [-2.28766266  4.80059714], Intensity: 47.49769246058423
Position: [-3.0529068   1.00831738], Intensity: 10.898052842545193
Position: [1.85683721 4.0203763 ], Intensity: 23.473242045519854
Position: [-0.5903136  2.0464193], Intensity: 

In [30]:
FA.optimize()
FA.get_optima()

(array([ 3.62694829, -1.43280871]), 7.944582261268458)

## TODO
Create another animation class to animate the optimization process

In [31]:
import pygame
import sys

class FFAnimation(Firefly_Optimization):
        
    def __init__(self, objective_fn, bounds, num_fireflies=10, 
                 iterations=10, alpha=1., beta_0=1., 
                 gamma=.01, is_minimization=True):
        super().__init__(objective_fn, bounds, num_fireflies, 
                 iterations, alpha, beta_0, 
                 gamma, is_minimization)
        

    def imize(self, a, screencale):
        return int((a - self.bounds[0][0]) / (self.bounds[0][1] - self.bounds[0][0]) * screencale), int((a - self.bounds[1][0]) / (self.bounds[1][1] - self.bounds[1][0]) * screencale)
        
        
    def draw(self, screen, scale):
        screen.fill((255, 255, 255))
        for firefly in self.fireflies:
            x, y = self.imize(firefly.position, scale)
            pygame.draw.circle(screen, (255, 0, 0), (x, y), 5)
        pygame.display.flip()
        

In [32]:
pygame.init()
screen_size = 600
screen = pygame.display.set_mode((screen_size, screen_size))
pygame.display.set_caption("FF Optimization")

ff = FFAnimation(objective_fn=rastrigin2d,bounds=bounds,num_fireflies=100, iterations=50)
ff.generate_population()
best_solution, best_fitness = ff.optimize()

print("\nBest Solution:", best_solution)
print("Best Fitness:", best_fitness)

#wait for close
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            sys.exit()


Best Solution: [-2.51153572 -3.26485662]
Best Fitness: 1.4094220449074797


SystemExit: 

c:\Users\Shira\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Exercises 

1. Rastrigin (https://www.sfu.ca/~ssurjano/rastr.html)
Styblinski-Tang (https://www.sfu.ca/~ssurjano/stybtang.html)
Eggholder (https://www.sfu.ca/~ssurjano/egg.html)
General functions available at: https://www.sfu.ca/~ssurjano/optimization.html
1. Extend to code so that it can use more than 2 dimensions. Try the Rastrigin-2,4 and 8. 
2. Modify the update equation so that instead of using a uniform random distribution, change it to another distribution such as normal distribution, levy distribution, etc. Experiments suggest that normal distribution gives better results. 
3. Modify code so that it terminates early if the best value does not change.
